# 07. Зведення результатів для звіту

Збираємо всі `runs/results/*.json` (на Kaggle з виходів ноутбуків 02–06, доданих як Input; локально ще й з `code/results/`) і будуємо таблиці та рисунки для `report.md`.
Public LB для кожного сабміту можна вписати вручну в `code/results/lb.csv` (колонки `name,public_lb`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ostapmn/ucu_ai_hw_2.git"
if Path("/kaggle/working").exists() and not Path("common.py").exists():
    repo = Path("/kaggle/working/repo")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
    os.chdir(repo / "code")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, ".")
DEBUG = os.environ.get("HW2_DEBUG") == "1"

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import analysis as an
import common as c

P = c.paths()
res = an.all_results(P).set_index("name")
lb_path = c.ROOT / "code" / "results" / "lb.csv"
if lb_path.exists():
    res = res.join(pd.read_csv(lb_path).set_index("name"), how="left")
print(len(res), "результатів")
res.family.value_counts()

## 1. Усі детектори на val

In [ ]:
ORDER = ["two-stage", "one-stage anchor-based", "one-stage anchor-free", "query-based", "budget-matched CNN", "pipeline"]
COLS = ["family", "mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "ms_per_img", "gpu_hours", "public_lb"]
models = res[res.family != "pipeline"].copy()
models["family"] = pd.Categorical(models.family, ORDER, ordered=True)
models = models.sort_values(["family", "mAP"])
table = models[[col for col in COLS if col in models.columns]]
md = an.md_table(table.drop(columns="family").round(3))
(P.results / "tables.md").write_text("## Моделі (val)\n\n" + md + "\n")
display(table.round(3))

In [ ]:
best = models.groupby("family", observed=True).mAP.idxmax()
sub = models.loc[best.values]
x = np.arange(3)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for i, (name, row) in enumerate(sub.iterrows()):
    axes[0].bar(x + i * 0.8 / len(sub), row[["AP_small", "AP_medium", "AP_large"]].astype(float), 0.8 / len(sub), label=name)
axes[0].set_xticks(x + 0.4 - 0.4 / len(sub), ["small", "medium", "large"])
axes[0].set_title("AP за розміром, найкраща модель кожного типу")
axes[0].legend(fontsize=8)
for fam, g in models.groupby("family", observed=True):
    axes[1].scatter(g.ms_per_img, g.mAP, label=fam)
    for name, row in g.iterrows():
        axes[1].annotate(name, (row.ms_per_img, row.mAP), fontsize=7, xytext=(3, 3), textcoords="offset points")
axes[1].set_xlabel("мс на кадр (batch 1)")
axes[1].set_ylabel("val mAP")
axes[1].set_title("точність vs швидкість")
axes[1].legend(fontsize=7)
fig.savefig(P.images / "summary_models.png", bbox_inches="tight", dpi=130)
plt.show()

In [ ]:
pc = sub[[f"AP_{k}" for k in c.CLASSES]].astype(float)
fig, ax = plt.subplots(figsize=(9, 0.5 * len(pc) + 1.5))
im = ax.imshow(pc.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(c.CLASSES)), c.CLASSES)
ax.set_yticks(range(len(pc)), pc.index)
for i in range(pc.shape[0]):
    for j in range(pc.shape[1]):
        ax.text(j, i, f"{pc.values[i, j]:.2f}", ha="center", va="center", color="w", fontsize=8)
ax.set_title("AP за класами")
fig.savefig(P.images / "summary_classes.png", bbox_inches="tight", dpi=130)
plt.show()

## 2. Фінальний пайплайн

In [ ]:
pipe = res[res.family == "pipeline"]
if len(pipe):
    t = pipe[[col for col in ["mAP", "AP50", "AP_small", "AP_medium", "AP_large", "public_lb"] if col in pipe.columns]]
    t.index = [i.removeprefix("pipe_") for i in t.index]
    with open(P.results / "tables.md", "a") as f:
        f.write("\n## Пайплайн (val)\n\n" + an.md_table(t.round(4)) + "\n")
    display(t.round(4))
print("таблиці для звіту:", P.results / "tables.md")